# Эксперимент: нейтрализация форматного конфаунда

EDA в~разделе~2.7 показал, что русские типографские маркеры (тире, кавычки-ёлочки, скобки) встречаются у~людей в~3,6--5,5 раза чаще, чем в~AI-текстах. TF-IDF на символьных $n$-граммах захватывает эти различия, и~часть высокого F1 модели~C может объясняться форматной асимметрией, а~не лексическим сигналом.

Эксперимент проверяет три конфигурации с~одинаковыми гиперпараметрами:

- **C** (orig→orig) — baseline, обучение и~тест на оригинальных данных
- **C'** (orig→norm) — adversarial-сценарий, обученная модель на нормализованных тестах
- **C''** (norm→norm) — академический контроль, обучение и~тест на нормализованных

Дельты:
- $C - C'$ — уязвимость модели к~runtime-нормализации (что произойдёт, если генератор завтра научится типографике)
- $C - C''$ — вклад форматного конфаунда в~сигнал
- $C' - C''$ — польза от знания о~нормализации при обучении

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

sys.path.insert(0, str(Path("..").resolve() / "scripts"))
from normalize_typography import count_typography_markers, normalize_typography

DATA_DIR = Path("..") / "data" / "splits"
NORM_DIR = DATA_DIR / "normalized"
NORM_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR = Path("..") / "artifacts"

SEED = 42

## Загрузка тематически согласованных сплитов

In [ ]:
train_tm = pd.read_json(DATA_DIR / "train_tm.jsonl", lines=True)
val_tm   = pd.read_json(DATA_DIR / "val_tm.jsonl", lines=True)
test_tm  = pd.read_json(DATA_DIR / "test_tm.jsonl", lines=True)
holdout_tm = pd.read_json(DATA_DIR / "holdout_tm_unseen.jsonl", lines=True)

test_tm_clean = test_tm[test_tm["source_model"] != "adversarial"].copy()
test_tm_adv   = test_tm[test_tm["source_model"] == "adversarial"].copy()

for name, df in [("train_tm", train_tm), ("val_tm", val_tm),
                 ("test_tm_clean", test_tm_clean), ("test_tm_adv", test_tm_adv),
                 ("holdout_tm", holdout_tm)]:
    print(f"{name:15s}: {len(df):>6,} rows")

## Аудит типографических маркеров по корпусу

Прогоняем `count_typography_markers` по всем текстам корпуса. Если в~топе появятся маркеры, не покрытые `normalize_typography`, до~эксперимента расширяем функцию.

In [ ]:
all_corpus = pd.concat([train_tm, val_tm, test_tm, holdout_tm], ignore_index=True)
audit = pd.DataFrame([count_typography_markers(t) for t in all_corpus["text"]])
audit_total = audit.sum().sort_values(ascending=False)
print("Маркеры по корпусу (по убыванию частоты):")
print(audit_total.to_string())
print(f"\nВсего текстов: {len(all_corpus):,}")
print(f"Текстов с хотя бы одним маркером: {(audit.sum(axis=1) > 0).sum():,}")

## Дифф 10 случайных текстов до/после нормализации

In [ ]:
sample = all_corpus.sample(n=10, random_state=SEED)
for i, (_, row) in enumerate(sample.iterrows(), 1):
    orig = row["text"]
    norm = normalize_typography(orig)
    if orig == norm:
        print(f"--- Sample {i} (label={row['label']}, len={len(orig)}): без изменений ---\n")
        continue
    print(f"--- Sample {i} (label={row['label']}, doc_id={row.get('doc_id', 'N/A')}) ---")
    print(f"BEFORE: {orig[:200]}")
    print(f"AFTER : {norm[:200]}")
    print()

## Нормализация всех сплитов и sanity-check

In [ ]:
def normalize_split(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["text"] = out["text"].apply(normalize_typography)
    return out

train_tm_n = normalize_split(train_tm)
val_tm_n   = normalize_split(val_tm)
test_tm_n  = normalize_split(test_tm)
holdout_tm_n = normalize_split(holdout_tm)
test_tm_clean_n = test_tm_n[test_tm_n["source_model"] != "adversarial"].copy()
test_tm_adv_n   = test_tm_n[test_tm_n["source_model"] == "adversarial"].copy()

# Sanity-check 1: средняя длина не упала больше чем на 2%
for name, orig, norm in [("train_tm", train_tm, train_tm_n),
                          ("test_tm", test_tm, test_tm_n),
                          ("holdout_tm", holdout_tm, holdout_tm_n)]:
    delta = (orig["text"].str.len().mean() - norm["text"].str.len().mean()) / orig["text"].str.len().mean()
    assert delta < 0.02, f"{name}: длина упала на {delta*100:.2f}% (>2%)"
    print(f"{name:15s}: длина упала на {delta*100:.3f}%")

# Sanity-check 2: doc_id сохранён построчно
for name, orig, norm in [("train_tm", train_tm, train_tm_n),
                          ("test_tm", test_tm, test_tm_n),
                          ("holdout_tm", holdout_tm, holdout_tm_n)]:
    assert (orig["doc_id"].values == norm["doc_id"].values).all(), f"{name}: doc_id mismatch"

# Sanity-check 3: метки классов сохранены
for name, orig, norm in [("train_tm", train_tm, train_tm_n),
                          ("test_tm", test_tm, test_tm_n),
                          ("holdout_tm", holdout_tm, holdout_tm_n)]:
    assert (orig["label"].values == norm["label"].values).all(), f"{name}: label mismatch"

print("\nВсе три проверки пройдены.")

In [ ]:
for name, df in [("train_tm", train_tm_n), ("val_tm", val_tm_n),
                 ("test_tm", test_tm_n), ("holdout_tm_unseen", holdout_tm_n)]:
    out = NORM_DIR / f"{name}.jsonl"
    df.to_json(out, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {out} ({len(df):,} rows)")

## Параметры моделей (зафиксированы)

Все три конфигурации используют одинаковые гиперпараметры (значения из раздела~2.5.2 и~notebook~03), чтобы изолировать эффект нормализации.

In [ ]:
TFIDF_CHAR_PARAMS = dict(
    analyzer="char_wb", ngram_range=(3, 5),
    max_features=30_000, sublinear_tf=True,
)
TFIDF_WORD_PARAMS = dict(
    analyzer="word", ngram_range=(1, 2),
    max_features=20_000, sublinear_tf=True,
    token_pattern=r"(?i)[а-яёa-z]{2,}",
)
LOGREG_PARAMS = dict(
    C=50.0, solver="saga", max_iter=1000, random_state=SEED,
)


def fit_tfidf_logreg(train_df, val_df, logreg_overrides=None):
    """Обучить TF-IDF (char+word) и LogReg на train+val."""
    tfidf_char = TfidfVectorizer(**TFIDF_CHAR_PARAMS)
    tfidf_word = TfidfVectorizer(**TFIDF_WORD_PARAMS)
    full = pd.concat([train_df, val_df], ignore_index=True)
    X_char = tfidf_char.fit_transform(full["text"])
    X_word = tfidf_word.fit_transform(full["text"])
    X = hstack([X_char, X_word])
    y = full["label"].values
    params = {**LOGREG_PARAMS, **(logreg_overrides or {})}
    model = LogisticRegression(**params).fit(X, y)
    return tfidf_char, tfidf_word, model


def evaluate(tfidf_char, tfidf_word, model, df):
    X = hstack([tfidf_char.transform(df["text"]), tfidf_word.transform(df["text"])])
    pred = model.predict(X)
    y = df["label"].values
    f1 = float(f1_score(y, pred, average="macro")) if len(np.unique(y)) > 1 else None
    recall_ai = float((pred[y == 1] == 1).mean()) if (y == 1).any() else None
    return {"f1_macro": f1, "recall_ai": recall_ai, "n": int(len(df))}

## Конфигурация C: orig→orig (baseline)

In [ ]:
print("Обучение C на оригинальных train_tm + val_tm...")
tfidf_char_C, tfidf_word_C, model_C = fit_tfidf_logreg(train_tm, val_tm)

C_results = {
    "test_tm":     evaluate(tfidf_char_C, tfidf_word_C, model_C, test_tm_clean),
    "holdout_tm":  evaluate(tfidf_char_C, tfidf_word_C, model_C, holdout_tm),
    "test_tm_adv": evaluate(tfidf_char_C, tfidf_word_C, model_C, test_tm_adv),
}
for k, v in C_results.items():
    print(f"  {k:14s}: {v}")

## Конфигурация C': orig→norm

Та~же модель `model_C` (обученная на оригинале) применяется через `transform()` к~нормализованным тестам. Никакого `fit` — иначе получили бы C''.

In [ ]:
C_prime_results = {
    "test_tm":     evaluate(tfidf_char_C, tfidf_word_C, model_C, test_tm_clean_n),
    "holdout_tm":  evaluate(tfidf_char_C, tfidf_word_C, model_C, holdout_tm_n),
    "test_tm_adv": evaluate(tfidf_char_C, tfidf_word_C, model_C, test_tm_adv_n),
}
for k, v in C_prime_results.items():
    print(f"  {k:14s}: {v}")

## Конфигурация C'': norm→norm (переобучение на нормализованных)

In [ ]:
print("Обучение C'' на нормализованных train_tm + val_tm...")
tfidf_char_Cpp, tfidf_word_Cpp, model_Cpp = fit_tfidf_logreg(train_tm_n, val_tm_n)

C_double_prime_results = {
    "test_tm":     evaluate(tfidf_char_Cpp, tfidf_word_Cpp, model_Cpp, test_tm_clean_n),
    "holdout_tm":  evaluate(tfidf_char_Cpp, tfidf_word_Cpp, model_Cpp, holdout_tm_n),
    "test_tm_adv": evaluate(tfidf_char_Cpp, tfidf_word_Cpp, model_Cpp, test_tm_adv_n),
}
for k, v in C_double_prime_results.items():
    print(f"  {k:14s}: {v}")

## Оценка случайного шума переобучения C''

Три прогона C'' с~разными seed для оценки нижней границы значимости дельт $C - C''$ и~$C' - C''$.

In [ ]:
sigma_runs = []
for seed in [42, 123, 7]:
    tc, tw, m = fit_tfidf_logreg(train_tm_n, val_tm_n, logreg_overrides={"random_state": seed})
    res = {
        "seed": seed,
        "test_tm_f1":    evaluate(tc, tw, m, test_tm_clean_n)["f1_macro"],
        "holdout_tm_f1": evaluate(tc, tw, m, holdout_tm_n)["f1_macro"],
        "adv_recall":    evaluate(tc, tw, m, test_tm_adv_n)["recall_ai"],
    }
    sigma_runs.append(res)
    print(f"seed={seed}: {res}")

sigma_df = pd.DataFrame(sigma_runs)
print("\nstd по трём прогонам:")
print(sigma_df[["test_tm_f1", "holdout_tm_f1", "adv_recall"]].std())

## Сводная таблица

In [ ]:
def fmt(v, digits=3):
    return f"{v:.{digits}f}" if v is not None else "—"

rows = []
for cfg_name, res in [("C (orig→orig)", C_results),
                      ("C' (orig→norm)", C_prime_results),
                      ("C'' (norm→norm)", C_double_prime_results)]:
    gap = (res["test_tm"]["f1_macro"] - res["holdout_tm"]["f1_macro"]) * 100
    rows.append({
        "Конфигурация": cfg_name,
        "F1 test_tm":     fmt(res["test_tm"]["f1_macro"]),
        "F1 holdout_tm":  fmt(res["holdout_tm"]["f1_macro"]),
        "Recall adv":     fmt(res["test_tm_adv"]["recall_ai"]),
        "Разрыв test−hold (п.п.)": f"{gap:.1f}",
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

# Дельты для интерпретации
print("\nДельты F1 (п.п.):")
print(f"  C - C'  на test_tm:     {(C_results['test_tm']['f1_macro'] - C_prime_results['test_tm']['f1_macro'])*100:+.2f}")
print(f"  C - C'  на holdout_tm:  {(C_results['holdout_tm']['f1_macro'] - C_prime_results['holdout_tm']['f1_macro'])*100:+.2f}")
print(f"  C - C'' на test_tm:     {(C_results['test_tm']['f1_macro'] - C_double_prime_results['test_tm']['f1_macro'])*100:+.2f}")
print(f"  C - C'' на holdout_tm:  {(C_results['holdout_tm']['f1_macro'] - C_double_prime_results['holdout_tm']['f1_macro'])*100:+.2f}")
print(f"  C' - C'' на test_tm:    {(C_prime_results['test_tm']['f1_macro'] - C_double_prime_results['test_tm']['f1_macro'])*100:+.2f}")
print(f"  C' - C'' на holdout_tm: {(C_prime_results['holdout_tm']['f1_macro'] - C_double_prime_results['holdout_tm']['f1_macro'])*100:+.2f}")

## Сохранение результатов

In [ ]:
output = {
    "experiment": "format_confound_ablation",
    "configurations": {
        "C_orig_to_orig":            C_results,
        "C_prime_orig_to_norm":      C_prime_results,
        "C_double_prime_norm_to_norm": C_double_prime_results,
    },
    "sigma_estimation": {
        "runs": sigma_runs,
        "std_test_tm_f1":    float(sigma_df["test_tm_f1"].std()),
        "std_holdout_tm_f1": float(sigma_df["holdout_tm_f1"].std()),
        "std_adv_recall":    float(sigma_df["adv_recall"].std()),
    },
    "audit_corpus_markers": audit_total.astype(int).to_dict(),
    "summary_table": summary.to_dict(orient="records"),
}

OUTPUT_PATH = ARTIFACTS_DIR / "format_confound_ablation.json"
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2, default=str)
print(f"Saved: {OUTPUT_PATH}")